# Notebook 3 — Feature Engineering
## AI Job Market Intelligence & Smart Recommender

**Objective:** Transform the cleaned dataset (`naukri_cleaned.csv`) into a modeling-ready feature matrix for the salary prediction model (Notebook 4), and produce a fully-grouped `analytics_jobs.csv` for the recommendation engine (Notebook 5).

**The single most important rule in this notebook:** the train/test split happens *first*, before any statistic (outlier bounds, "top N" category lists, TF-IDF vocabulary) is computed. Every such statistic is derived from the training split only, then applied — not recomputed — on the test split. This notebook exists partly to correct a real leakage bug from an earlier version of this project, where outlier bounds and top-N category selection were computed on the full dataset *before* splitting, meaning test-set values silently influenced decisions about the training data. That bug is fixed here, and this note stays in the notebook so the reasoning isn't lost.

**Scope:**
1. Filter to rows usable for salary modeling
2. Train/test split (before any leakage-prone step)
3. Outlier bounds — derived from train, applied to both
4. Deterministic feature transforms (log target, experience buckets, city/industry normalization) — safe to apply identically everywhere since they don't depend on the full data's distribution
5. Group definitions (top cities, top industries) — derived from train, applied to both, and also applied to the full dataset for the recommendation engine
6. Multicollinearity check (VIF) — actually run, not just imported and ignored
7. TF-IDF — fit on train only
8. Final feature matrix assembly and artifact saving

**Reference:** `docs/02_Data_Dictionary.md` for all column definitions.


## 1. Import Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from statsmodels.stats.outliers_influence import variance_inflation_factor


## 2. Load Cleaned Data

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "dataset" / "naukri_cleaned.csv"
ANALYTICS_OUTPUT_PATH = PROJECT_ROOT / "dataset" / "analytics_jobs.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()


Shape: (30000, 20)


,uniq_id,crawl_timestamp,job_title,job_salary,job_experience_required,key_skills,role_category,location,functional_area,industry,role,salary_disclosed,salary_min,salary_max,salary_avg,exp_min,exp_max,exp_avg,location_cleaned,key_skills_cleaned
0,9be62c49a0b7ebe982a4af1edaa7bc5f,2019-07-05 01:46:07+00:00,Digital Media Planner,Not Disclosed by Recruiter,5 - 10 yrs,Media Planning| Digital Media,Advertising,Mumbai,"Marketing , Advertising , MR , PR , Media Plan...","Advertising, PR, MR, Event Management",Media Planning Executive/Manager,0,NaN,NaN,NaN,5.0,10.0,7.5,Mumbai,"media planning, digital media"
1,3c52d436e39f596b22519da2612f6a56,2019-07-06 08:04:50+00:00,Online Bidding Executive,Not Disclosed by Recruiter,2 - 5 yrs,pre sales| closing| software knowledge| clien...,Retail Sales,"Pune,Pune","Sales , Retail , Business Development","IT-Software, Software Services",Sales Executive/Officer,0,NaN,NaN,NaN,2.0,5.0,3.5,Pune,"pre sales, closing, software knowledge, client..."
2,ffad8a2396c60be2bf6d0e2ff47c58d4,2019-08-05 15:50:44+00:00,Trainee Research/ Research Executive- Hi- Tec...,Not Disclosed by Recruiter,0 - 1 yrs,Computer science| Fabrication| Quality check|...,R&D,Gurgaon,"Engineering Design , R&D","Recruitment, Staffing",R&D Executive,0,NaN,NaN,NaN,0.0,1.0,0.5,Gurgaon,"computer science, fabrication, quality check, ..."
3,7b921f51b5c2fb862b4a5f7a54c37f75,2019-08-05 15:31:56+00:00,Technical Support,"2,00,000 - 4,00,000 PA.",0 - 5 yrs,Technical Support,Admin/Maintenance/Security/Datawarehousing,Mumbai,"IT Software - Application Programming , Mainte...","IT-Software, Software Services",Technical Support Engineer,1,200000.0,400000.0,300000.0,0.0,5.0,2.5,Mumbai,technical support
4,2d8b7d44e138a54d5dc841163138de50,2019-07-05 02:48:29+00:00,Software Test Engineer -hyderabad,Not Disclosed by Recruiter,2 - 5 yrs,manual testing| test engineering| test cases|...,Programming & Design,Hyderabad,IT Software - QA & Testing,"IT-Software, Software Services",Testing Engineer,0,NaN,NaN,NaN,2.0,5.0,3.5,Hyderabad,"manual testing, test engineering, test cases, ..."


## 3. Filter to Model-Usable Rows

The salary model can only be trained on rows with a parsed salary *and* a parsed experience value (both are required inputs). This is the selection-bias point flagged repeatedly in Notebooks 1–2 and in the BRD: **the resulting training population is not a random sample of all job postings** — it's the subset that disclosed a parseable salary. That limitation is restated here because it directly shapes every decision in this notebook, and again in Notebook 4 where the model's real-world applicability is assessed.

In [3]:
modeling_df = df[
    (df["salary_disclosed"] == 1) &
    (df["salary_avg"].notna()) &
    (df["exp_avg"].notna())
].copy()

print(f"Full cleaned dataset: {len(df):,} rows")
print(f"Model-usable subset (parsed salary + parsed experience): {len(modeling_df):,} rows "
      f"({len(modeling_df)/len(df):.1%} of the full dataset)")


Full cleaned dataset: 30,000 rows
Model-usable subset (parsed salary + parsed experience): 7,734 rows (25.8% of the full dataset)


## 4. Train / Test Split — Done FIRST

Every downstream statistic in this notebook (outlier bounds, top-N city/industry lists, TF-IDF vocabulary) must be derived from `train` only. Splitting here, before those steps, is what makes that possible — this ordering is the actual fix for the leakage bug in the original version of this notebook.

In [4]:
train_df, test_df = train_test_split(
    modeling_df,
    test_size=0.2,
    random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)


Train shape: (6187, 20)
Test shape: (1547, 20)


## 5. Salary Outlier Bounds — Derived From Train Only

The 1st/99th percentile bounds are computed from `train_df["salary_avg"]` only. The *same* numeric bounds (not recomputed) are then used to filter both train and test — this is the leakage-safe version of the outlier removal that, in the original version of this project, was computed on the full pre-split dataset.

In [5]:
lower_bound = train_df["salary_avg"].quantile(0.01)
upper_bound = train_df["salary_avg"].quantile(0.99)

print(f"Train-derived salary bounds: Rs. {lower_bound:,.0f} - Rs. {upper_bound:,.0f}")

train_before = len(train_df)
test_before = len(test_df)

train_df = train_df[(train_df["salary_avg"] >= lower_bound) & (train_df["salary_avg"] <= upper_bound)].copy()
test_df = test_df[(test_df["salary_avg"] >= lower_bound) & (test_df["salary_avg"] <= upper_bound)].copy()

print(f"Train: {train_before} -> {len(train_df)} rows ({train_before - len(train_df)} removed)")
print(f"Test:  {test_before} -> {len(test_df)} rows ({test_before - len(test_df)} removed)")


Train-derived salary bounds: Rs. 95,000 - Rs. 3,250,000
Train: 6187 -> 6068 rows (119 removed)
Test:  1547 -> 1516 rows (31 removed)


In [6]:
train_df["log_salary"] = np.log1p(train_df["salary_avg"])
test_df["log_salary"] = np.log1p(test_df["salary_avg"])


## 6. Deterministic Feature Transforms

The following transforms are pure functions of a single row's existing values — they don't depend on the distribution of any other row, so applying them independently to train, test, and later the full dataset carries no leakage risk.

### 6.1 Experience Engineering

In [7]:
def add_experience_features(data):
    data = data.copy()
    data["exp_squared"] = data["exp_avg"] ** 2

    def exp_bucket(x):
        if pd.isna(x):
            return "Unknown"
        elif x <= 2:
            return "Entry"
        elif x <= 5:
            return "Mid"
        elif x <= 10:
            return "Senior"
        else:
            return "Leadership"

    data["exp_bucket"] = data["exp_avg"].apply(exp_bucket)
    return data


train_df = add_experience_features(train_df)
test_df = add_experience_features(test_df)


### 6.2 City Normalization

In [8]:
CITY_MAP = {
    "New Delhi": "Delhi",
    "Delhi NCR": "Delhi",
    "Noida": "Delhi",
    "Gurgaon": "Delhi",
    "Navi Mumbai": "Mumbai",
}


def normalize_city(data):
    data = data.copy()
    data["city_cleaned"] = data["location_cleaned"].replace(CITY_MAP)
    return data


train_df = normalize_city(train_df)
test_df = normalize_city(test_df)


### 6.3 Industry Contamination Filter

In [9]:
def filter_contaminated_industry(data):
    """Removes rows where `industry` is actually a company-description paragraph
    rather than a category label, as identified and quantified in Notebook 2."""
    data = data.copy()
    data = data[data["industry"].str.len() < 100].copy()
    data["industry"] = data["industry"].str.strip().str.lower()
    return data


before_train, before_test = len(train_df), len(test_df)
train_df = filter_contaminated_industry(train_df)
test_df = filter_contaminated_industry(test_df)

print(f"Train: removed {before_train - len(train_df)} contaminated-industry rows")
print(f"Test:  removed {before_test - len(test_df)} contaminated-industry rows")


Train: removed 6 contaminated-industry rows
Test:  removed 1 contaminated-industry rows


### 6.4 Role Grouping

In [10]:
def role_group(role):
    role = str(role).lower()
    if "developer" in role or "engineer" in role or "architect" in role:
        return "Tech"
    elif "sales" in role or "business development" in role:
        return "Sales"
    elif "hr" in role or "recruit" in role:
        return "HR"
    elif "finance" in role or "account" in role:
        return "Finance"
    elif "manager" in role or "operations" in role:
        return "Operations"
    else:
        return "Other"


train_df["role_group"] = train_df["role"].apply(role_group)
test_df["role_group"] = test_df["role"].apply(role_group)

coverage = (train_df["role_group"] != "Other").mean()
print(f"Role grouping coverage on train: {coverage:.1%} assigned to a specific bucket, "
      f"{1-coverage:.1%} fell into 'Other'")
print(train_df["role_group"].value_counts())


Role grouping coverage on train: 54.8% assigned to a specific bucket, 45.2% fell into 'Other'
role_group
Other         2742
Tech          1118
Sales          946
Operations     632
Finance        317
HR             307
Name: count, dtype: int64


**Note:** the 'Other' bucket size is reported explicitly above rather than left unexamined — if it were the majority class, the rule-based grouping would need rework before being trusted as a model feature. The printed distribution is the actual check.

## 7. Group Definitions Derived From Train Only

Unlike the transforms above, "top N cities" and "top N industries" are *not* pure per-row functions — they depend on the frequency distribution across the whole dataset. Computing them from `train_df` only, then applying the same fixed group membership to `test_df`, is what avoids leaking test-set category frequency information into the training data. Any city or industry in the test set that wasn't in train's top-N list is correctly bucketed as "Other," exactly as it would be for a genuinely new/unseen value in production.

In [11]:
TOP_N_CITIES = 7
TOP_N_INDUSTRIES = 10

top_cities = train_df["city_cleaned"].value_counts().nlargest(TOP_N_CITIES).index
top_industries = train_df["industry"].value_counts().nlargest(TOP_N_INDUSTRIES).index

print("Top cities (from train):", list(top_cities))
print("Top industries (from train):", list(top_industries))


Top cities (from train): ['Delhi', 'Mumbai', 'Bengaluru', 'Pune', 'Hyderabad', 'Chennai', 'Kolkata']
Top industries (from train): ['it-software, software services', 'bpo, call centre, ites', 'banking, financial services, broking', 'education, teaching, training', 'other', 'recruitment, staffing', 'medical, healthcare, hospitals', 'travel , hotels , restaurants , airlines , railways', 'automobile, auto anciliary, auto components', 'insurance']


In [12]:
def apply_city_group(data):
    data = data.copy()
    data["city_group"] = data["city_cleaned"].apply(lambda x: x if x in top_cities else "Other")
    return data


def apply_industry_group(data):
    data = data.copy()
    data["industry_group"] = data["industry"].apply(lambda x: x if x in top_industries else "Other")
    return data


train_df = apply_city_group(train_df)
test_df = apply_city_group(test_df)
train_df = apply_industry_group(train_df)
test_df = apply_industry_group(test_df)


### Combined Industry × Role Label

`industry_role` is a human-readable combined category, kept for **display purposes in the recommendation engine only** (Notebook 5) — e.g. "it-software_Tech". In the original version of this project it was computed but never actually included in the model's feature set, which is dead code left in silently. That's fixed here by being explicit: this column is not part of `X_train_final` / `X_test_final` below, and is documented as display-only.

In [13]:
train_df["industry_role"] = train_df["industry_group"] + "_" + train_df["role_group"]
test_df["industry_role"] = test_df["industry_group"] + "_" + test_df["role_group"]


## 8. Interaction Features — Removed, With Reasoning

The original version of this notebook created `exp_city_interaction` and `exp_industry_interaction` by multiplying `exp_avg` by `.cat.codes` — an arbitrary integer assigned to each category based on alphabetical order (e.g. "Delhi" = 3, "Mumbai" = 5 purely because of sort order). The resulting feature has no real-world interpretation: multiplying experience by an arbitrary label ID is not a meaningful quantity, and a technical interviewer asking "what does this feature represent" would not have a defensible answer.

**Decision:** these two features are removed rather than fixed, for two reasons:
1. Both models used downstream (Random Forest, XGBoost) are tree-based ensembles, which learn feature interactions natively by splitting on combinations of features — they don't need a hand-engineered interaction term to find "high experience + metro city" patterns; the model can already discover that from `exp_avg` and the one-hot city columns directly.
2. A defensible numeric interaction (e.g. target-encoded city × experience) would require computing a per-category target statistic in a way that avoids leakage — solvable, but adds real complexity for uncertain benefit here, and is better left as documented future work than shipped in a form that can't be explained.

This is a case where the correct fix is *removal with a documented reason*, not a patch — and that reasoning is the answer to give if asked about it directly.

## 9. Multicollinearity Check (VIF)

The original notebook imported `variance_inflation_factor` and never used it — despite creating `exp_squared`, a polynomial term that is, by construction, correlated with `exp_avg`. Actually running the check here, rather than leaving the import as unused boilerplate.

In [14]:
from statsmodels.tools.tools import add_constant

vif_features = train_df[["exp_avg", "exp_squared"]].dropna()
vif_features_with_const = add_constant(vif_features)

vif_data = pd.DataFrame()
vif_data["feature"] = vif_features_with_const.columns
vif_data["VIF"] = [
    variance_inflation_factor(vif_features_with_const.values, i)
    for i in range(vif_features_with_const.shape[1])
]

print(vif_data)


       feature       VIF
0        const  5.575724
1      exp_avg  7.108164
2  exp_squared  7.108164


**Interpretation:** `exp_avg` and `exp_squared` show meaningful VIF above the common rule-of-thumb threshold of 5–10 — expected, since `exp_squared` is a deterministic function of `exp_avg` by construction (this is the standard behavior of polynomial features, not a data quality problem).

**Decision:** both are kept anyway. VIF/multicollinearity is a concern for **linear models**, where it inflates coefficient variance and makes individual coefficients uninterpretable. It is not a material concern for the **tree-based models** (Random Forest, XGBoost) that are the primary models in Notebook 4 — trees split on one feature at a time and are not sensitive to linear correlation between inputs the way OLS coefficients are. The Linear Regression baseline in Notebook 4 is evaluated on predictive performance only, not on interpreting its individual coefficients, so the collinearity here doesn't undermine any claim actually being made from it. This reasoning is stated explicitly rather than leaving the VIF import unused and unexplained, as in the original version.

## 10. Train/Test Split of Features (X/y) — Already Split Above; Finalizing Column Sets

In [15]:
X_train = train_df.drop(columns=["log_salary"])
y_train = train_df["log_salary"].reset_index(drop=True)

X_test = test_df.drop(columns=["log_salary"])
y_test = test_df["log_salary"].reset_index(drop=True)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)


## 11. Skill Text (TF-IDF) — Fit on Train Only

This step was already leakage-safe in the original version of this project — TF-IDF is correctly fit only on the training split's skill text and applied (not refit) to test. Kept as-is; included here for completeness of the pipeline.

In [16]:
X_train["key_skills_cleaned"] = X_train["key_skills_cleaned"].fillna("")
X_test["key_skills_cleaned"] = X_test["key_skills_cleaned"].fillna("")

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9+.# ]", " ", text)
    return text

X_train["skills_cleaned"] = X_train["key_skills_cleaned"].apply(clean_text)
X_test["skills_cleaned"] = X_test["key_skills_cleaned"].apply(clean_text)

tfidf = TfidfVectorizer(max_features=300, stop_words="english")

train_tfidf = tfidf.fit_transform(X_train["skills_cleaned"])
test_tfidf = tfidf.transform(X_test["skills_cleaned"])

print("TF-IDF vocabulary size:", len(tfidf.get_feature_names_out()))


TF-IDF vocabulary size: 300


## 12. Categorical Encoding

In [17]:
cat_cols = ["exp_bucket", "city_group", "industry_group", "role_group"]

X_train_cat = pd.get_dummies(X_train[cat_cols], drop_first=True)
X_test_cat = pd.get_dummies(X_test[cat_cols], drop_first=True)

# Align test columns to train columns exactly — any category present in test
# but not in train's dummy set is implicitly zero (correct: it means that
# category was never seen during training, so the model has no learned
# coefficient/split for it anyway).
X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)

print("Categorical feature count:", X_train_cat.shape[1])


Categorical feature count: 25


## 13. Assemble Final Feature Matrix

In [18]:
num_cols = ["exp_avg", "exp_squared"]

X_train_num = X_train[num_cols].reset_index(drop=True)
X_test_num = X_test[num_cols].reset_index(drop=True)

train_tfidf_df = pd.DataFrame(train_tfidf.toarray(), columns=tfidf.get_feature_names_out())
test_tfidf_df = pd.DataFrame(test_tfidf.toarray(), columns=tfidf.get_feature_names_out())

X_train_final = pd.concat(
    [X_train_num, X_train_cat.reset_index(drop=True), train_tfidf_df],
    axis=1
)

X_test_final = pd.concat(
    [X_test_num, X_test_cat.reset_index(drop=True), test_tfidf_df],
    axis=1
)

print("Final train feature matrix:", X_train_final.shape)
print("Final test feature matrix:", X_test_final.shape)

assert list(X_train_final.columns) == list(X_test_final.columns), "Train/test feature columns must match exactly"
print("Column alignment check passed: train and test feature sets are identical.")


Final train feature matrix: (6062, 327)
Final test feature matrix: (1515, 327)
Column alignment check passed: train and test feature sets are identical.


## 14. Build the Full Analytics Dataset (for the Recommendation Engine)

The recommendation engine (Notebook 5) needs the **entire** job pool — including postings that never disclosed a salary — not just the model-training subset. The same deterministic transforms and the same train-derived group definitions (`top_cities`, `top_industries`) are applied to the full cleaned dataset here, so that a job's `city_group`/`industry_group` label means exactly the same thing whether it came from the training data or from a job the model never saw.

In [19]:
analytics_df = df.copy()

analytics_df = add_experience_features(analytics_df)
analytics_df = normalize_city(analytics_df)
analytics_df = filter_contaminated_industry(analytics_df)
analytics_df = apply_city_group(analytics_df)
analytics_df = apply_industry_group(analytics_df)
analytics_df["role_group"] = analytics_df["role"].apply(role_group)
analytics_df["industry_role"] = analytics_df["industry_group"] + "_" + analytics_df["role_group"]

analytics_df = analytics_df[[
    "uniq_id",
    "job_title",
    "industry",
    "industry_group",
    "role",
    "role_group",
    "industry_role",
    "city_cleaned",
    "city_group",
    "salary_disclosed",
    "salary_min",
    "salary_max",
    "salary_avg",
    "exp_avg",
    "exp_bucket",
    "key_skills_cleaned",
]].copy()

print("Analytics dataset shape:", analytics_df.shape)
analytics_df.head()


Analytics dataset shape: (29408, 16)


,uniq_id,job_title,industry,industry_group,role,role_group,industry_role,city_cleaned,city_group,salary_disclosed,salary_min,salary_max,salary_avg,exp_avg,exp_bucket,key_skills_cleaned
0,9be62c49a0b7ebe982a4af1edaa7bc5f,Digital Media Planner,"advertising, pr, mr, event management",Other,Media Planning Executive/Manager,Operations,Other_Operations,Mumbai,Mumbai,0,NaN,NaN,NaN,7.5,Senior,"media planning, digital media"
1,3c52d436e39f596b22519da2612f6a56,Online Bidding Executive,"it-software, software services","it-software, software services",Sales Executive/Officer,Sales,"it-software, software services_Sales",Pune,Pune,0,NaN,NaN,NaN,3.5,Mid,"pre sales, closing, software knowledge, client..."
2,ffad8a2396c60be2bf6d0e2ff47c58d4,Trainee Research/ Research Executive- Hi- Tec...,"recruitment, staffing","recruitment, staffing",R&D Executive,Other,"recruitment, staffing_Other",Delhi,Delhi,0,NaN,NaN,NaN,0.5,Entry,"computer science, fabrication, quality check, ..."
3,7b921f51b5c2fb862b4a5f7a54c37f75,Technical Support,"it-software, software services","it-software, software services",Technical Support Engineer,Tech,"it-software, software services_Tech",Mumbai,Mumbai,1,200000.0,400000.0,300000.0,2.5,Mid,technical support
4,2d8b7d44e138a54d5dc841163138de50,Software Test Engineer -hyderabad,"it-software, software services","it-software, software services",Testing Engineer,Tech,"it-software, software services_Tech",Hyderabad,Hyderabad,0,NaN,NaN,NaN,3.5,Mid,"manual testing, test engineering, test cases, ..."


**Note:** this dataset is intentionally *not* filtered to salary-disclosed rows — the recommendation engine needs to recommend from the full job pool, including postings with unknown salary (handled explicitly in Notebook 5's scoring logic, not silently dropped).

In [20]:
analytics_df.to_csv(ANALYTICS_OUTPUT_PATH, index=False)
print(f"Analytics dataset saved to: {ANALYTICS_OUTPUT_PATH}")


Analytics dataset saved to: c:\Users\venut\OneDrive\Desktop\job_AI_system\dataset\analytics_jobs.csv


## 15. Save Modeling Artifacts

In [21]:
MODELS_DATA_DIR = PROJECT_ROOT / "dataset"
MODELS_DATA_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(X_train_final, MODELS_DATA_DIR / "X_train.pkl")
joblib.dump(X_test_final, MODELS_DATA_DIR / "X_test.pkl")
joblib.dump(y_train, MODELS_DATA_DIR / "y_train.pkl")
joblib.dump(y_test, MODELS_DATA_DIR / "y_test.pkl")
joblib.dump(tfidf, MODELS_DATA_DIR / "tfidf_vectorizer.pkl")

# Group-definition artifacts: saved so any future/new data can be assigned to the
# SAME city/industry groups the model was trained on, without needing to re-derive
# them (and without accidentally re-deriving them from a different, potentially
# leaked, population). This was not saved at all in the original version of this
# project — a real reproducibility gap, fixed here.
group_mappings = {
    "top_cities": list(top_cities),
    "top_industries": list(top_industries),
    "city_map": CITY_MAP,
    "salary_lower_bound": float(lower_bound),
    "salary_upper_bound": float(upper_bound),
}
joblib.dump(group_mappings, MODELS_DATA_DIR / "group_mappings.pkl")

print("Feature engineering artifacts saved successfully.")
print("Saved:", [p.name for p in MODELS_DATA_DIR.glob("*.pkl")])


Feature engineering artifacts saved successfully.
Saved: ['group_mappings.pkl', 'tfidf_vectorizer.pkl', 'X_test.pkl', 'X_train.pkl', 'y_test.pkl', 'y_train.pkl']


## Summary

**What changed vs. the original version of this notebook, and why:**

| Issue | Fix |
|---|---|
| Outlier bounds, top-N city/industry selection computed on full data before split | Train/test split now happens first; every such statistic is derived from train only |
| `exp_city_interaction` / `exp_industry_interaction` used arbitrary `.cat.codes` with no real meaning | Removed, with documented reasoning (tree models learn interactions natively; a defensible alternative would need leakage-safe target encoding, left as future work) |
| `variance_inflation_factor` imported but never called | Actually run on `exp_avg`/`exp_squared`; high VIF noted and explicitly justified as expected-and-acceptable given the tree-based models used downstream |
| `industry_role` computed but silently unused in the model (dead code) | Kept only as a documented display-only column for the recommendation engine, explicitly excluded from the model feature set |
| No group-mapping artifacts saved (top cities/industries were implicit, not persisted) | Saved as `group_mappings.pkl` for reproducibility |
| Role-grouping "Other" bucket size never reported | Printed explicitly (Section 6.4) as a real check, not assumed acceptable |

**Known limitations carried forward:**
- The salary model's training population remains the disclosed-salary subset — a non-random sample, as stated since Notebook 1. Nothing in this notebook fixes that; it's a property of the source data, not of this pipeline, and must be restated in Notebook 4.
- Role grouping is rule-based keyword matching, not learned — the "Other" bucket size (reported above) is the actual measure of how much this simplification costs.

**Next notebook:** `4_salary_prediction_model.ipynb` — train and evaluate the salary model on `X_train_final` / `X_test_final`.
